# 2 · Evaluating a strategy

Is my strategy any good — and how would I know?

The short answer is that a single run cannot tell you. This notebook builds
up to the comparison that can.

In [1]:
import pretium as pt

universe = pt.Universe.random(40, seed=111)
print("universe:", len(universe), "instruments")

universe: 40 instruments


## A strategy as data

`StrategySpec` writes a strategy down as a declarative, versioned, hashable
document instead of an arbitrary Python callable. That matters for a reason
that is not tidiness: a result which depends on a callable cannot be cited.
A reader could re-run your seed and get your market, then have no way to get
your *strategy*.

The grammar is deliberately small — a signal, a concentration (`top_k`), an
exposure (`gross`) and a participation cap.

In [2]:
spec = pt.StrategySpec.momentum(lookback_days=1.0, top_k=5)

print(spec.to_json())
print("fingerprint:", spec.fingerprint)

{
  "spec_version": 1,
  "signal": {
    "kind": "momentum",
    "lookback_days": 1.0
  },
  "portfolio": {
    "gross": 1.0,
    "top_k": 5
  },
  "execution": {
    "cadence": "step",
    "max_participation": 0.02
  },
  "seed": null
}
fingerprint: e6bbc35c6f0968b1f178e1f7ee926d449a8d3fa72440e476dd8b25c7a6a50895


What it deliberately **cannot** express: path dependence (stop losses,
drawdown limits, anything reading its own P&L history), conditional logic,
and custom signals. Those need a Python agent — which works everywhere a
spec does, at the cost that the result is not citable as a spec and the
methods section has to cite code at a commit.

## One market, every strategy

`evaluate` runs every entrant against an *identical* market. That is what
makes the comparison exact — and it is also why one run is not an answer.

In [3]:
entrants = {"mine": spec}
entrants.update(pt.baselines.reference_agents(seed=7))

scores = pt.evaluate(entrants, seed=7, universe=universe, days=10)

print(f"{'agent':16s} {'return':>9s} {'trades':>7s} {'impact bps':>11s}")
for name, s in sorted(scores.items(), key=lambda kv: -kv[1].return_pct):
    print(f"{name:16s} {s.return_pct:8.3f}% {s.trades:7d} {s.impact_bps:11.2f}")

agent               return  trades  impact bps
mean_reversion     12.609%     801       26.74
oracle              8.052%     576       11.49
mine                0.598%     700       50.92
momentum            0.598%     700       50.92
buy_and_hold       -0.233%      40       11.09
random             -1.791%    2387       10.16


The baselines are there on purpose. A return of +4% means nothing without
knowing what buy-and-hold did on the same market.

## Capture: the ceiling, not the target

`oracle` reads the simulator's own fair value — it can see what no real
trader can. Capture ratio expresses P&L as a fraction of that ceiling.

In [4]:
capture = pt.capture_ratio(scores)
for name, value in sorted(capture.items(), key=lambda kv: -kv[1]):
    print(f"  {name:16s} {value:7.3f}")

  mean_reversion     1.566
  mine               0.074
  momentum           0.074
  buy_and_hold      -0.029
  random            -0.222


1.0 would be perfect foresight. It is a scale, not a goal.

## One seed measures the seed

Here is the same comparison on three different market draws. Watch the
ordering.

In [5]:
for seed in (7, 8, 9):
    e = {"mine": spec}
    e.update(pt.baselines.reference_agents(seed=seed))
    s = pt.evaluate(e, seed=seed, universe=universe, days=10)
    ranked = sorted(s.items(), key=lambda kv: -kv[1].return_pct)
    print(f"seed {seed}: " + "  ".join(f"{n}({v.return_pct:+.1f}%)"
                                        for n, v in ranked[:4]))

seed 7: mean_reversion(+12.6%)  oracle(+8.1%)  mine(+0.6%)  momentum(+0.6%)


seed 8: oracle(+7.3%)  mean_reversion(+6.7%)  buy_and_hold(+0.9%)  random(+0.1%)


seed 9: oracle(+12.6%)  mine(+5.0%)  momentum(+5.0%)  mean_reversion(+4.1%)


If the ordering moved, you have just seen why a single-seed leaderboard is a
measurement of the seed.

## The version worth believing

`rank` runs many seeds and compares entrants **pairwise on the same market
draw**. Pairing removes the market from the question, which is what makes a
small sample informative.

It takes a *factory* rather than built agents, deliberately: agents are
stateful, and a reused instance carries one market's history into the next
with no visible symptom. A spec is immune because it is rebuilt each time.

In [6]:
def make_agents():
    e = {"mine": pt.StrategySpec.momentum(lookback_days=1.0, top_k=5)}
    e.update(pt.baselines.reference_agents(seed=0))
    return e

ranking = pt.rank(make_agents, seeds=[1, 2, 3, 4, 5, 6],
                  universe=universe, days=5)

print(f"{'agent':16s} {'pooled capture':>15s} {'median P&L':>13s} {'first on':>9s}")
for r in ranking.table():
    pooled = "n/a" if r.pooled_capture is None else f"{r.pooled_capture:.3f}"
    print(f"{r.name:16s} {pooled:>15s} {r.median_pnl:13,.0f} "
          f"{r.wins:5d}/{len(r.measured)}")

agent             pooled capture    median P&L  first on
mean_reversion             0.551        33,609     6/6
mine                      -0.044            30     0/6
momentum                  -0.044            30     0/6
random                    -0.056        -2,932     0/6
buy_and_hold              -0.397       -30,944     0/6


Quote **`pooled_capture`** — total P&L over the reference's total. The
`first on` column counts seeds where an entrant ranked first among *all*
entrants, which is a league position rather than a head-to-head record and
splits arbitrarily on a tie.

## Is A really better than B?

A paired sign test. `decisive` is true only when one entrant won on every
paired seed — the strongest claim a sign test can make, and the only one
needing no distributional assumption.

In [7]:
for other in ("buy_and_hold", "random", "mean_reversion"):
    t = ranking.separation("mine", other)
    p = "n/a" if t["p_value"] is None else f"{t['p_value']:.3f}"
    print(f"  mine vs {other:16s} {t['wins_a']}-{t['wins_b']}"
          f"  ties {t['ties']}  decisive={t['decisive']}  p={p}")

  mine vs buy_and_hold     4-2  ties 0  decisive=False  p=0.688
  mine vs random           3-3  ties 0  decisive=False  p=1.000
  mine vs mean_reversion   0-6  ties 0  decisive=True  p=0.031


`unmeasurable` names entrants the test could not separate. That is a real
answer, not a gap in the data.

In [8]:
print("unmeasurable:", list(ranking.unmeasurable) or "none")
print()
print(ranking.report())

unmeasurable: none

6 seeds on universe 5d8de78b55aa... under model pt-v3
  mean_reversion    capture +0.551  per-seed [+0.218, +1.286]  wins 6/6
  mine              capture -0.044  per-seed [-0.395, +0.214]  wins 0/6
  momentum          capture -0.044  per-seed [-0.395, +0.214]  wins 0/6
  random            capture -0.056  per-seed [-0.120, +0.015]  wins 0/6
  buy_and_hold      capture -0.397  per-seed [-0.822, +0.283]  wins 0/6


## Read the failures, not the P&L

Good results here do not predict real returns — the price process comes from
a known model, and a strategy that fits its structure will look brilliant
and teach you nothing. A strategy that *fails* here has still told you
something: it broke against a live order book under honest impact costs.

Next: **[3 · Why did the price move](03-why-did-the-price-move.ipynb)**.